# RECAP Ablation Study — Prefix-side Advantage Conditioning

Five ablation training runs to characterize the design space:

| Run | Ablation | Change vs main |
|-----|----------|----------------|
| A1 | Main (reference) | full setup, weighted sampler, action expert trainable |
| A2 | Uniform sampler | sampler weights = 1.0 (no class balancing) |
| A3 | Token-only training | only advantage token trainable, action expert frozen |
| A4 | LR low | learning rate = 5e-5 |
| A5 | LR high | learning rate = 5e-4 |

Each run trains 800 steps at batch 16. Total time: ~2.5 hours.
Outputs saved to `/mnt/rollouts/ablations/<run_name>/`.

All runs logged to W&B project `IDL_34_RECAP`.

**Method**: prefix-side advantage token injection. A learnable embedding for
A_pos / A_neg / A_null is prepended to the VLM context, conditioning the
flow-matching action expert on trajectory quality.


## Step 1 — Environment

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
DRIVE_ROOT = "/mnt/rollouts"
os.environ["HF_HOME"]         = f"{DRIVE_ROOT}/hf_cache"
os.environ["HF_LEROBOT_HOME"] = f"{DRIVE_ROOT}/lerobot_cache"


In [ ]:
!apt-get install -y -qq libegl1-mesa-dev libgl1-mesa-dev libgles2-mesa-dev > /dev/null 2>&1
!pip install -q "lerobot[smolvla,libero]" wandb scikit-learn


In [3]:
import torch
print(f"CUDA: {torch.cuda.is_available()}  |  {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")


CUDA: True  |  NVIDIA A100-SXM4-40GB


## Step 2 — Weights & Biases setup

Run `!wandb login` once interactively, or paste your API key into the env var below.
Each ablation run logs to project `IDL_34_RECAP` with a descriptive run name.


In [ ]:
import wandb


WANDB_PROJECT = "IDL_34_RECAP"
WANDB_ENTITY  = "idl_34"
WANDB_MODE    = "online"

WANDB_BASE_CONFIG = {
    "model":             "SmolVLA-LIBERO",
    "method":            "prefix-RECAP",
    "task_suite":        "libero_spatial",
    "n_tasks":           10,
    "advantage_classes": 3,
    "injection_point":   "prefix",
}

print(f"W&B project: {WANDB_PROJECT}, mode: {WANDB_MODE}")


## Step 3 — Constants and paths

In [5]:
from pathlib import Path
import numpy as np

ROLLOUT_DATASET_DIR = Path(DRIVE_ROOT) / "rollouts_with_labels"
ABLATION_ROOT       = Path(DRIVE_ROOT) / "ablations_prefix"
ABLATION_ROOT.mkdir(parents=True, exist_ok=True)

BC_POLICY_REPO  = "HuggingFaceVLA/smolvla_libero"
LIBERO_SUITE    = "libero_spatial"
ADV_NEG, ADV_POS, ADV_NULL = 0, 1, 2

TASK_DESCRIPTIONS = {
    0: "pick up the black bowl between the plate and the ramekin and place it on the plate",
    1: "pick up the black bowl from table center and place it on the plate",
    2: "pick up the black bowl in the top drawer of the wooden cabinet and place it on the plate",
    3: "pick up the black bowl next to the cookie box and place it on the plate",
    4: "pick up the black bowl next to the plate and place it on the plate",
    5: "pick up the black bowl next to the ramekin and place it on the plate",
    6: "pick up the black bowl on the cookie box and place it on the plate",
    7: "pick up the black bowl on the ramekin and place it on the plate",
    8: "pick up the black bowl on the stove and place it on the plate",
    9: "pick up the black bowl on the wooden cabinet and place it on the plate",
}

assert ROLLOUT_DATASET_DIR.exists(), f"Rollout dataset not found at {ROLLOUT_DATASET_DIR}"
print(f"ROLLOUT_DATASET_DIR = {ROLLOUT_DATASET_DIR}")
print(f"ABLATION_ROOT       = {ABLATION_ROOT}")


ROLLOUT_DATASET_DIR = /mnt/rollouts/rollouts_with_labels
ABLATION_ROOT       = /mnt/rollouts/ablations_prefix


## Step 4 — LIBERO config

In [6]:
import yaml
import libero

libero_config_dir = Path.home() / ".libero"
libero_config_dir.mkdir(parents=True, exist_ok=True)
libero_data_root = Path(DRIVE_ROOT) / "libero_data"
libero_data_root.mkdir(parents=True, exist_ok=True)
pkg_dir = Path(libero.__file__).parent / "libero"
with open(libero_config_dir / "config.yaml", "w") as f:
    yaml.safe_dump({
        "benchmark_root": str(pkg_dir),
        "bddl_files":     str(pkg_dir / "bddl_files"),
        "init_states":    str(pkg_dir / "init_files"),
        "datasets":       str(libero_data_root),
        "assets":         str(pkg_dir / "assets"),
    }, f)
print("LIBERO config written.")


LIBERO config written.


## Step 5 — Prefix advantage embedding module + patch

The advantage token is a learnable embedding sized to the VLM hidden dim
(SmolVLM2 uses 960). At training time, we prepend the advantage token to the
VLM prefix sequence by patching `embed_prefix`.

Note on inference: at eval time, the advantage token gets prepended to the
prefix and baked into the KV cache. Setting `use_cache=False` would force
the prefix to be recomputed every denoising step (10x slower but consistent
with training). For these training-only ablations, KV cache behavior at
inference doesn't affect what we measure.


In [7]:
import torch.nn as nn
import torch.nn.functional as F


class PrefixAdvantageEmbedding(nn.Module):
    def __init__(self, vlm_hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(3, vlm_hidden_size)
        nn.init.normal_(self.embedding.weight, mean=0.0, std=0.02)
        with torch.no_grad():
            self.embedding.weight[ADV_NULL].zero_()

    def forward(self, advantage_labels):
        return self.embedding(advantage_labels)


def _patched_embed_prefix(self, images, img_masks, lang_tokens, lang_masks, state=None):
    embs, pad_masks, att_masks = self.__class__._orig_embed_prefix(
        self, images, img_masks, lang_tokens, lang_masks, state=state)

    label = getattr(self, "_train_advantage_label",
                    getattr(self, "_eval_advantage_label", ADV_POS))
    bs = embs.shape[0]
    if isinstance(label, torch.Tensor):
        adv_labels = label.to(embs.device)
    else:
        adv_labels = torch.full((bs,), int(label), dtype=torch.long, device=embs.device)

    adv_emb = self.advantage_token(adv_labels).to(embs.dtype)
    adv_emb = adv_emb.unsqueeze(1)

    new_embs      = torch.cat([adv_emb, embs], dim=1)
    new_pad_masks = torch.cat(
        [torch.ones(bs, 1, dtype=pad_masks.dtype, device=pad_masks.device), pad_masks], dim=1)
    new_att_masks = torch.cat(
        [torch.zeros(bs, 1, dtype=att_masks.dtype, device=att_masks.device), att_masks], dim=1)

    return new_embs, new_pad_masks, new_att_masks


print("Prefix advantage module + patch defined.")


Prefix advantage module + patch defined.


## Step 6 — Load policy, attach advantage token, install patch

Each ablation run reloads from `BC_POLICY_REPO` to avoid state leakage between runs.


In [ ]:
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy, make_att_2d_masks

device = torch.device("cuda")


def fresh_policy():
    pol = SmolVLAPolicy.from_pretrained(BC_POLICY_REPO).to(device)
    vlm_hidden = pol.model.vlm_with_expert.config.text_config.hidden_size
    pol.model.advantage_token = PrefixAdvantageEmbedding(vlm_hidden).to(device)
    cls = type(pol.model)
    if not hasattr(cls, "_orig_embed_prefix"):
        cls._orig_embed_prefix = cls.embed_prefix
    cls.embed_prefix = _patched_embed_prefix
    return pol


_test = fresh_policy()
VLM_HIDDEN     = _test.model.vlm_with_expert.config.text_config.hidden_size
CHUNK_SIZE     = _test.config.chunk_size
MAX_ACTION_DIM = _test.config.max_action_dim
tokenizer      = _test.model.vlm_with_expert.processor.tokenizer
print(f"vlm_hidden_size  = {VLM_HIDDEN}")
print(f"CHUNK_SIZE = {CHUNK_SIZE}, MAX_ACTION_DIM = {MAX_ACTION_DIM}")
del _test
torch.cuda.empty_cache()


## Step 7 — Datasets (rollouts + expert)

In [9]:
import glob
import pandas as pd
from torch.utils.data import Dataset


def _unpack(x):
    out = []
    stack = [x]
    while stack:
        item = stack.pop()
        if isinstance(item, (list, tuple, np.ndarray)):
            stack.extend(reversed(list(item)) if hasattr(item, "__iter__") else [item])
        else:
            out.append(item)
    return out


class RolloutFrameDataset(Dataset):
    def __init__(self, root):
        self.files = sorted(glob.glob(str(Path(root) / "data" / "chunk-000" / "file-*.parquet")))
        assert self.files, f"No parquet files in {root}/data/chunk-000/"
        self._labels, self._file_idx, self._row_idx, self._task_idx = [], [], [], []
        tasks_pq = Path(root) / "meta" / "tasks.parquet"
        tdf = pd.read_parquet(tasks_pq)
        self._tasks_map = dict(zip(tdf["task_index"].tolist(), tdf.index.tolist()))
        print(f"Indexing {len(self.files)} parquet files ...")
        for fi, f in enumerate(self.files):
            df = pd.read_parquet(f, columns=["advantage_label", "task_index"])
            for r in range(len(df)):
                lbl = df["advantage_label"].iat[r]
                ti  = df["task_index"].iat[r]
                self._labels.append(int(lbl[0]) if hasattr(lbl, "__len__") else int(lbl))
                self._task_idx.append(int(ti[0]) if hasattr(ti, "__len__") else int(ti))
                self._file_idx.append(fi)
                self._row_idx.append(r)
        self._labels = np.array(self._labels, dtype=np.int64)
        self._task_idx = np.array(self._task_idx, dtype=np.int64)
        self._cache = {}
        print(f"Rollout frames: {len(self._labels)}  "
              f"(pos={(self._labels==1).sum()}, neg={(self._labels==0).sum()})")

    def __len__(self):
        return len(self._labels)

    def _load_file(self, fi):
        if fi in self._cache:
            return self._cache[fi]
        df = pd.read_parquet(self.files[fi],
                             columns=["observation.images.image",
                                      "observation.images.image2",
                                      "observation.state",
                                      "action"])
        if len(self._cache) >= 4:
            self._cache.pop(next(iter(self._cache)))
        self._cache[fi] = df
        return df

    def __getitem__(self, i):
        df = self._load_file(self._file_idx[i])
        r = self._row_idx[i]
        img1   = np.asarray(_unpack(df["observation.images.image"].iat[r]),  dtype=np.float32).reshape(3, 256, 256)
        img2   = np.asarray(_unpack(df["observation.images.image2"].iat[r]), dtype=np.float32).reshape(3, 256, 256)
        state  = np.asarray(_unpack(df["observation.state"].iat[r]), dtype=np.float32)
        action = np.asarray(_unpack(df["action"].iat[r]), dtype=np.float32)
        return {
            "observation.images.image":  torch.from_numpy(img1),
            "observation.images.image2": torch.from_numpy(img2),
            "observation.state":         torch.from_numpy(state),
            "action":                    torch.from_numpy(action),
            "advantage_label":           int(self._labels[i]),
            "task":                      self._tasks_map.get(int(self._task_idx[i]), ""),
        }


rollout_ds = RolloutFrameDataset(ROLLOUT_DATASET_DIR)


Indexing 47 parquet files ...
Rollout frames: 15462  (pos=7875, neg=7587)


In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

print("Loading expert dataset (cached if previously downloaded) ...")
expert_ds = LeRobotDataset("HuggingFaceVLA/libero")
print(f"Expert dataset: {expert_ds.meta.total_episodes} episodes, {len(expert_ds)} frames")

SPATIAL_TASK_STRINGS = set(TASK_DESCRIPTIONS.values())
spatial_episodes = []
for ep_i in range(expert_ds.meta.total_episodes):
    ep = expert_ds.meta.episodes[ep_i]
    if len(ep["tasks"]) == 1 and ep["tasks"][0] in SPATIAL_TASK_STRINGS:
        spatial_episodes.append((ep_i, ep["dataset_from_index"], ep["dataset_to_index"], ep["tasks"][0]))

print(f"Spatial episodes: {len(spatial_episodes)}")


class ExpertSpatialDataset(Dataset):
    def __init__(self, ds, spatial_episodes):
        self.ds = ds
        self.frame_map = []
        for _, src, dst, task_str in spatial_episodes:
            for f in range(src, dst):
                self.frame_map.append((f, task_str))

    def __len__(self):
        return len(self.frame_map)

    def __getitem__(self, i):
        ds_idx, task_str = self.frame_map[i]
        s = self.ds[ds_idx]
        return {
            "observation.images.image":  s["observation.images.image"],
            "observation.images.image2": s["observation.images.image2"],
            "observation.state":         s["observation.state"],
            "action":                    s["action"],
            "advantage_label":           1,
            "task":                      task_str,
        }


expert_spatial_ds = ExpertSpatialDataset(expert_ds, spatial_episodes)
print(f"Expert Spatial dataset: {len(expert_spatial_ds)} frames")


## Step 8 — Training infrastructure (loss + loop)

In [11]:
import json, time
from torch.utils.data import DataLoader, ConcatDataset, WeightedRandomSampler
from tqdm.auto import tqdm
from lerobot.utils.constants import ACTION


def collate(bl):
    return {
        "observation.images.image":  torch.stack([b["observation.images.image"] for b in bl]),
        "observation.images.image2": torch.stack([b["observation.images.image2"] for b in bl]),
        "observation.state":         torch.stack([b["observation.state"] for b in bl]),
        "action":                    torch.stack([b["action"] for b in bl]),
        "advantage_label":           torch.tensor([int(b["advantage_label"]) for b in bl], dtype=torch.long),
        "task":                      [b["task"] for b in bl],
    }


def batch_to_device(batch, dev):
    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            batch[k] = v.to(dev, non_blocking=True)
    return batch


def recap_loss_prefix(policy, batch, adv):
    bs  = batch[ACTION].shape[0]
    dev = adv.device
    images, img_masks = policy.prepare_images(batch)
    state   = policy.prepare_state(batch)
    actions = policy.prepare_action(batch)
    tok = tokenizer(list(batch["task"]), padding="max_length",
                    max_length=policy.config.tokenizer_max_length,
                    truncation=True, return_tensors="pt")
    lang_tokens = tok["input_ids"].to(dev)
    lang_masks  = tok["attention_mask"].to(dev).bool()

    if actions.ndim == 2:
        actions = actions.unsqueeze(1).expand(-1, CHUNK_SIZE, -1)
    if actions.shape[-1] < MAX_ACTION_DIM:
        actions = F.pad(actions, (0, MAX_ACTION_DIM - actions.shape[-1]))

    noise = policy.model.sample_noise(actions.shape, dev)
    t     = policy.model.sample_time(bs, dev)
    x_t   = t[:, None, None] * noise + (1.0 - t[:, None, None]) * actions
    u_t   = noise - actions

    policy.model._train_advantage_label = adv

    prefix_embs, prefix_pad, _ = policy.model.embed_prefix(
        images, img_masks, lang_tokens, lang_masks, state=state)
    suffix_embs, suffix_pad, _ = policy.model.embed_suffix(x_t, t)

    pad = torch.cat([prefix_pad, suffix_pad], dim=1)
    p_len, s_len = prefix_pad.shape[1], suffix_pad.shape[1]
    att = torch.cat([torch.zeros(bs, p_len, dtype=torch.bool, device=dev),
                     torch.ones(bs, s_len, dtype=torch.bool, device=dev)], dim=1)
    att_2d  = make_att_2d_masks(pad, att)
    pos_ids = torch.cumsum(pad, dim=1) - 1

    (_, suffix_out), _ = policy.model.vlm_with_expert.forward(
        attention_mask=att_2d, position_ids=pos_ids,
        past_key_values=None, inputs_embeds=[prefix_embs, suffix_embs],
        use_cache=False, fill_kv_cache=False)
    suffix_out = suffix_out[:, -CHUNK_SIZE:].to(dtype=torch.float32)
    v_t = policy.model.action_out_proj(suffix_out)

    real_dim = policy.config.action_feature.shape[0]
    per = F.mse_loss(u_t[..., :real_dim], v_t[..., :real_dim], reduction="none")
    return per.flatten(1).mean(dim=1)


def configure_trainable(policy, mode):
    if mode == "full":
        keywords = ("advantage_token", "action_in_proj", "action_out_proj",
                    "action_time_mlp", "state_proj", "lm_expert")
    elif mode == "token_only":
        keywords = ("advantage_token",)
    else:
        raise ValueError(mode)
    for name, p in policy.named_parameters():
        p.requires_grad = any(k in name for k in keywords)
    n_trn = sum(p.numel() for p in policy.parameters() if p.requires_grad)
    n_tot = sum(p.numel() for p in policy.parameters())
    return n_trn, n_tot


def make_sampler(rollout_ds, expert_size, mode):
    n_rollout_pos = int((rollout_ds._labels == 1).sum())
    n_rollout_neg = int((rollout_ds._labels == 0).sum())
    total_pos = expert_size + n_rollout_pos
    total_neg = n_rollout_neg
    total     = expert_size + len(rollout_ds)
    weights   = np.empty(total, dtype=np.float64)
    if mode == "weighted":
        w_pos = 1.0 / total_pos
        w_neg = 1.0 / total_neg
        weights[:expert_size] = w_pos
        for i in range(len(rollout_ds)):
            weights[expert_size + i] = w_pos if rollout_ds._labels[i] == 1 else w_neg
    elif mode == "uniform":
        weights[:] = 1.0
    else:
        raise ValueError(mode)
    return WeightedRandomSampler(weights=weights, num_samples=total, replacement=True)


def train_ablation(run_name, *, lr=1e-4, sampler_mode="weighted",
                   trainable_mode="full", steps=800, batch=16,
                   log_every=50, save_weights=False):
    print(f"\n{'='*70}\n  {run_name}\n{'='*70}")
    run_dir = ABLATION_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    cfg = {
        "run_name":       run_name,
        "lr":             lr,
        "sampler_mode":   sampler_mode,
        "trainable_mode": trainable_mode,
        "steps":          steps,
        "batch":          batch,
        **WANDB_BASE_CONFIG,
    }

    wb_run = wandb.init(
        project=WANDB_PROJECT, entity=WANDB_ENTITY, mode=WANDB_MODE,
        name=run_name, config=cfg, reinit=True,
    )

    policy = fresh_policy()
    n_trn, n_tot = configure_trainable(policy, trainable_mode)
    print(f"Trainable: {n_trn/1e6:.1f}M / {n_tot/1e6:.1f}M")
    wb_run.summary["trainable_params_M"] = round(n_trn / 1e6, 2)

    mixed_ds = ConcatDataset([expert_spatial_ds, rollout_ds])
    sampler  = make_sampler(rollout_ds, len(expert_spatial_ds), sampler_mode)
    dl = DataLoader(mixed_ds, batch_size=batch, sampler=sampler,
                    num_workers=4, pin_memory=True, drop_last=True,
                    collate_fn=collate, persistent_workers=True, prefetch_factor=2)

    optim = torch.optim.AdamW(
        [p for p in policy.parameters() if p.requires_grad],
        lr=lr, weight_decay=1e-10, betas=(0.9, 0.95))

    policy.train()
    history = []
    running_tot = running_pos = running_neg = 0.0
    n_pos_b = n_neg_b = 0
    step = 0
    t0 = time.time()
    pbar = tqdm(total=steps, desc=run_name, dynamic_ncols=True)

    while step < steps:
        for batch_d in dl:
            if step >= steps:
                break
            batch_d = batch_to_device(batch_d, device)
            adv = batch_d["advantage_label"].to(device).long()
            per_sample = recap_loss_prefix(policy, batch_d, adv)
            loss = per_sample.mean()
            optim.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=10.0)
            optim.step()

            with torch.no_grad():
                pm = (adv == 1); nm = (adv == 0)
                if pm.any():
                    running_pos += per_sample[pm].mean().item(); n_pos_b += 1
                if nm.any():
                    running_neg += per_sample[nm].mean().item(); n_neg_b += 1
                running_tot += loss.item()

            step += 1
            pbar.update(1)
            pbar.set_postfix({"loss": f"{loss.item():.3f}"})

            if step % log_every == 0:
                p = running_pos / max(n_pos_b, 1)
                n = running_neg / max(n_neg_b, 1)
                t = running_tot / log_every
                d = p - n
                tqdm.write(f"[{run_name} step {step:5d}] "
                           f"loss={t:.4f} pos={p:.4f} neg={n:.4f} Δ={d:+.4f} "
                           f"elapsed={(time.time()-t0)/60:.1f}m")
                history.append((step, t, p, n))
                wb_run.log({
                    "step":        step,
                    "loss/total":  t,
                    "loss/pos":    p,
                    "loss/neg":    n,
                    "loss/delta":  d,
                    "elapsed_min": (time.time() - t0) / 60,
                })
                running_tot = running_pos = running_neg = 0.0
                n_pos_b = n_neg_b = 0

    pbar.close()
    elapsed = (time.time() - t0) / 60
    final_delta = history[-1][2] - history[-1][3] if history else 0.0
    print(f"\n  done in {elapsed:.1f} min — final Δ = {final_delta:+.4f}")
    wb_run.summary["final_delta"]      = float(final_delta)
    wb_run.summary["final_total_loss"] = history[-1][1] if history else 0.0
    wb_run.summary["elapsed_min"]      = elapsed

    with open(run_dir / "loss_history.json", "w") as f:
        json.dump(history, f)
    adv_state = {k: v.cpu() for k, v in policy.model.advantage_token.state_dict().items()}
    torch.save(adv_state, run_dir / "advantage_token.pt")
    if save_weights:
        policy.save_pretrained(run_dir)

    wb_run.finish()
    del policy
    torch.cuda.empty_cache()
    return history, run_dir


## Step 9 — Run the five ablations sequentially

Order is intentional: A1 first (saves the reference advantage embedding), then variants.


In [ ]:
ABLATIONS = [
    dict(run_name="A1_main",       lr=1e-4, sampler_mode="weighted", trainable_mode="full",       save_weights=False),
    dict(run_name="A2_uniform",    lr=1e-4, sampler_mode="uniform",  trainable_mode="full",       save_weights=False),
    dict(run_name="A3_token_only", lr=1e-4, sampler_mode="weighted", trainable_mode="token_only", save_weights=False),
    dict(run_name="A4_lr_low",     lr=5e-5, sampler_mode="weighted", trainable_mode="full",       save_weights=False),
    dict(run_name="A5_lr_high",    lr=5e-4, sampler_mode="weighted", trainable_mode="full",       save_weights=False),
]

STEPS = 800
results = {}
for cfg in ABLATIONS[4:5]:
    hist, rdir = train_ablation(steps=STEPS, **cfg)
    results[cfg["run_name"]] = {"history": hist, "dir": str(rdir)}

print("\n\nAll ablations done.")
print(f"Results saved to {ABLATION_ROOT}")


## Step 10 — Compare loss curves across ablations

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
colors = {"A1_main": "#2a9d8f", "A2_uniform": "#e76f51", "A3_token_only": "#9b59b6",
          "A4_lr_low": "#f4a261", "A5_lr_high": "#264653"}

for name, data in results.items():
    h = np.array(data["history"])
    if len(h) == 0:
        continue
    axes[0].plot(h[:, 0], h[:, 1], label=name, color=colors.get(name, "gray"), linewidth=2)
    delta = h[:, 2] - h[:, 3]
    axes[1].plot(h[:, 0], delta, label=name, color=colors.get(name, "gray"), linewidth=2)

axes[0].set_xlabel("step"); axes[0].set_ylabel("total loss")
axes[0].set_title("Total flow-matching loss across ablations")
axes[0].legend(loc="upper right"); axes[0].grid(alpha=0.3)

axes[1].axhline(0, color="gray", linestyle="--", alpha=0.5)
axes[1].set_xlabel("step"); axes[1].set_ylabel("Δ = pos loss − neg loss")
axes[1].set_title("Advantage differentiation (Δ) across ablations")
axes[1].legend(loc="upper right"); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(ABLATION_ROOT / "ablation_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

sm = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, mode=WANDB_MODE,
                name="comparison_summary", reinit=True)
sm.log({"comparison": wandb.Image(str(ABLATION_ROOT / "ablation_comparison.png"))})
sm.finish()
print(f"Saved comparison to {ABLATION_ROOT / 'ablation_comparison.png'}")


## Step 11 — Summary table

In [18]:
summary = []
for name, data in results.items():
    h = data["history"]
    if not h:
        continue
    final_total = h[-1][1]; final_pos = h[-1][2]; final_neg = h[-1][3]
    final_delta = final_pos - final_neg
    summary.append({
        "run":         name,
        "final_total": round(final_total, 4),
        "final_pos":   round(final_pos, 4),
        "final_neg":   round(final_neg, 4),
        "final_delta": round(final_delta, 4),
    })

print(f"{'Run':<18} {'Total':>8} {'Pos':>8} {'Neg':>8} {'Δ':>10}")
print("-" * 56)
for s in summary:
    print(f"{s['run']:<18} {s['final_total']:>8.4f} {s['final_pos']:>8.4f} "
          f"{s['final_neg']:>8.4f} {s['final_delta']:>+10.4f}")

with open(ABLATION_ROOT / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"\nSaved summary to {ABLATION_ROOT / 'summary.json'}")


Run                   Total      Pos      Neg          Δ
--------------------------------------------------------
A5_lr_high           0.1038   0.0956   0.1118    -0.0162

Saved summary to /mnt/rollouts/ablations_prefix/summary.json


## Step 12 — Advantage embedding analysis (uses A1_main)

Loads the trained advantage embedding from `A1_main` and projects the three
embeddings to 2D via PCA. Pairwise cosine similarities tell us if the tokens
learned distinct directions in embedding space.


In [ ]:
from sklearn.decomposition import PCA
ref_path = ABLATION_ROOT / "A1_main" / "advantage_token.pt"
state = torch.load(ref_path, map_location="cpu")
emb = state["embedding.weight"].numpy()  # (3, vlm_hidden)

print(f"Advantage embeddings shape: {emb.shape}")
print(f"  A_neg  norm: {np.linalg.norm(emb[ADV_NEG]):.4f}")
print(f"  A_pos  norm: {np.linalg.norm(emb[ADV_POS]):.4f}")
print(f"  A_null norm: {np.linalg.norm(emb[ADV_NULL]):.4f}")


def cos(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))


print(f"\nPairwise cosine similarities:")
print(f"  pos · neg  = {cos(emb[ADV_POS], emb[ADV_NEG]):+.4f}")
print(f"  pos · null = {cos(emb[ADV_POS], emb[ADV_NULL]):+.4f}")
print(f"  neg · null = {cos(emb[ADV_NEG], emb[ADV_NULL]):+.4f}")

pca = PCA(n_components=2)
emb_2d = pca.fit_transform(emb)

plt.figure(figsize=(7, 6))
labels = ["A_neg", "A_pos", "A_null"]
colors_pts = ["#e76f51", "#2a9d8f", "#888888"]
for i, (lbl, col) in enumerate(zip(labels, colors_pts)):
    plt.scatter(emb_2d[i, 0], emb_2d[i, 1], s=300, color=col, label=lbl, edgecolors="black")
    plt.annotate(lbl, (emb_2d[i, 0], emb_2d[i, 1]), fontsize=12,
                 xytext=(10, 10), textcoords="offset points")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
plt.title("Advantage token embeddings — PCA projection (A1_main)")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(ABLATION_ROOT / "embedding_pca.png", dpi=150, bbox_inches="tight")
plt.show()

sm = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, mode=WANDB_MODE,
                name="embedding_analysis", reinit=True)
sm.log({"embedding_pca": wandb.Image(str(ABLATION_ROOT / "embedding_pca.png"))})
sm.finish()
